## Grocery App Endpoint - Filtered Products

We are building a grocery app. The first endpoint should return **filtered products** from the inventory.

Given the inventory, write `filter_products(filters)` that returns products matching the filter criteria.

### Input Filter Shape
`filters` can contain:
- `"itemType"`: category name (example: `"produce"`)
- `"qualities"`: list of qualities/tags (example: `["organic"]`)

### Behavior
- `{ "itemType": "produce" }` returns all produce products.
- `{ "itemType": "produce", "qualities": ["organic"] }` returns only organic produce.
- `{ "qualities": ["organic"] }` returns all organic products across categories.

### Rules
- Filters are **AND**, not OR.
- Matching is case-insensitive.
- Return only product names (do not return categories).
- Keep the output deterministic by sorting product names alphabetically.
- If no filters are provided, return all product names.

In [1]:
def filter_products(filters: dict) -> list:
    inventory = {
        "Produce": {
            "Apple": ["Gala", "organic"],
            "Pear": ["Bosc", "organic"],
            "Banana": [],
        },
        "Protein": {
            "Beef": ["tenderloin"],
            "beef": ["filet mignon"],
            "pork": ["loin chop", "organic"],
            "tofu": ["firm"],
        },
        "Grain": {
            "Wheat Flour": [],
            "Corn Flour": [],
            "Rice Flour": [],
            "Rice": ["organic"],
        },
        "Condiments": {"Soy Sauce": [], "Mustard": []},
    }

    filters = filters or {}
    item_type = (filters.get("itemType") or "").lower()
    qualities = [q.lower() for q in (filters.get("qualities") or [])]

    result = []
    for category, products in inventory.items():
        if item_type and category.lower() != item_type:
            continue

        for product, tags in products.items():
            tags_lower = [t.lower() for t in tags]

            matches_all_qualities = True
            for quality in qualities:
                if quality not in tags_lower:
                    matches_all_qualities = False
                    break

            if not matches_all_qualities:
                continue

            result.append(product)

    return sorted(result, key=str.lower)


res = filter_products({"itemType": "produce", "qualities": ["organic"]})
print(res)

['Apple', 'Pear']


In [3]:
import torch
def slow_pairwise_l2(x, y):
    # x: [N, D], y: [M, D]
    out = torch.zeros(x.shape[0], y.shape[0], device=x.device)

    for i in range(x.shape[0]):
        for j in range(y.shape[0]):
            out[i, j] = torch.sum((x[i] - y[j]) ** 2)

    return out

def pairwise_l2(x, y):
    """
    x: [N, D]
    y: [M, D]
    returns: [N, M]
    """
    x2 = (x ** 2).sum(dim=1, keepdim=True)       # [N, 1]
    y2 = (y ** 2).sum(dim=1).unsqueeze(0)        # [1, M]
    xy = x @ y.T                                # [N, M]

    dist = x2 + y2 - 2 * xy
    return dist.clamp_min(0.0)


# ----- Create example data -----
torch.manual_seed(42)   # for reproducibility
N, M, D = 4, 5, 3       # small sizes for easy inspection

x = torch.randn(N, D)   # e.g., [[a1,a2,a3], ...]
y = torch.randn(M, D)

# ----- Test both versions -----
print("x:\n", x)
print("\ny:\n", y)
print("\n" + "="*50)

# Slow version (explicit loops)
out_slow = slow_pairwise_l2(x, y)
print("Slow pairwise L2:\n", out_slow)

# Fast algebraic version
out_fast = pairwise_l2(x, y)
print("\nFast pairwise L2:\n", out_fast)

# Verify they match (within tolerance)
diff = torch.abs(out_slow - out_fast).max()
print(f"\nMax absolute difference: {diff:.6f}")
print("Results match!" if diff < 1e-5 else "Mismatch found!")


x:
 tensor([[ 0.3367,  0.1288,  0.2345],
        [ 0.2303, -1.1229, -0.1863],
        [ 2.2082, -0.6380,  0.4617],
        [ 0.2674,  0.5349,  0.8094]])

y:
 tensor([[ 1.1103, -1.6898, -0.9890],
        [ 0.9580,  1.3221,  0.8172],
        [-0.7658, -0.7506,  1.3525],
        [ 0.6863, -0.3278,  0.7950],
        [ 0.2815,  0.0562,  0.5227]])

Slow pairwise L2:
 tensor([[5.4026, 2.1496, 3.2391, 0.6449, 0.0914],
        [1.7400, 7.5145, 3.4991, 1.8031, 1.8955],
        [4.4160, 5.5316, 9.6513, 2.5235, 4.1977],
        [8.8938, 1.0968, 3.0151, 0.9199, 0.3116]])

Fast pairwise L2:
 tensor([[5.4026, 2.1496, 3.2391, 0.6449, 0.0914],
        [1.7400, 7.5145, 3.4991, 1.8031, 1.8955],
        [4.4160, 5.5316, 9.6513, 2.5235, 4.1977],
        [8.8938, 1.0968, 3.0151, 0.9199, 0.3116]])

Max absolute difference: 0.000001
Results match!
